In [ ]:
import sys
from pathlib import Path

# Permite importar a pasta src a partir do notebook
sys.path.append(str(Path.cwd().parent))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, recall_score
from src.models.oee import calcular_oee_predito, PENALIDADES_OEE

sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.float_format", lambda x: "%.4f" % x)

# Carga dos dados tratados
caminho_dados = Path("../data/processed/dados_sensores_tratados.csv")
df = pd.read_csv(caminho_dados)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values(by="timestamp").reset_index(drop=True)

print(f"Total de registros: {len(df)} | Período: {df['timestamp'].min()} até {df['timestamp'].max()}")

In [ ]:
# Split estritamente temporal (ordem cronológica para séries industriais)
split_idx = int(len(df) * 0.70)
df_train = df.iloc[:split_idx].copy()
df_test = df.iloc[split_idx:].copy()

print(f"Treino (70%): {df_train.shape[0]} amostras | {df_train['timestamp'].min()} -> {df_train['timestamp'].max()}")
print(f"Teste  (30%): {df_test.shape[0]} amostras | {df_test['timestamp'].min()} -> {df_test['timestamp'].max()}")

# Seleção de features descartando metadados e o alvo
colunas_excluir = ["timestamp", "id_celula", "liga_metalica", "tipo_falha"]
features = [col for col in df.columns if col not in colunas_excluir]

X_train = df_train[features]
y_train = df_train["tipo_falha"]
X_test = df_test[features]
y_test = df_test["tipo_falha"]

print(f"\nTotal de features preditivas: {len(features)}")
print("Distribuição de classes no Treino (%):")
print((y_train.value_counts(normalize=True) * 100).round(2))

In [ ]:
# 1. Random Forest com ponderação de classes
rf = RandomForestClassifier(
    n_estimators=150,
    max_depth=12,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

# 2. HistGradientBoosting com ponderação de classes
hgb = HistGradientBoostingClassifier(
    class_weight="balanced",
    max_iter=150,
    learning_rate=0.08,
    random_state=42
)
hgb.fit(X_train, y_train)
y_pred_hgb = hgb.predict(X_test)

nomes_classes = [PENALIDADES_OEE[i]["descricao"] for i in range(4)]

print("=" * 60)
print("RELATÓRIO: RANDOM FOREST CLASSIFIER")
print("=" * 60)
print(classification_report(y_test, y_pred_rf, target_names=nomes_classes, digits=4))

print("=" * 60)
print("RELATÓRIO: HIST GRADIENT BOOSTING")
print("=" * 60)
print(classification_report(y_test, y_pred_hgb, target_names=nomes_classes, digits=4))

In [ ]:
# Avaliação da exigência da rubrica de avaliação
recalls = recall_score(y_test, y_pred_hgb, average=None)
print("--- Validação do Critério de Excelência (Recall >= 90%) ---")
for i, r in enumerate(recalls):
    aprovado = "APROVADO" if r >= 0.90 else "REPROVADO"
    print(f"Classe {i} ({nomes_classes[i]}): Recall = {r * 100:.2f}% -> {aprovado}")

# Plotagem das Matrizes de Confusão
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, preds, tit in zip(axes, [y_pred_rf, y_pred_hgb], ["Random Forest", "HistGradientBoosting"]):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=range(4), yticklabels=range(4), ax=ax)
    ax.set_title(f"Matriz de Confusão: {tit}")
    ax.set_xlabel("Predito")
    ax.set_ylabel("Real")
plt.tight_layout()
plt.show()

In [ ]:
# Aplica a regra de OEE sobre as previsões do modelo de teste
df_test_resultado = df_test.copy()
df_test_resultado["tipo_falha_predito"] = y_pred_hgb

df_oee = calcular_oee_predito(df_test_resultado, col_classe_predita="tipo_falha_predito")

resumo_oee = df_oee.groupby("id_celula")[["oee_disp", "oee_perf", "oee_qual", "oee_global"]].mean() * 100
print("--- OEE Médio Estimado por Célula no Conjunto de Teste (%) ---")
display(resumo_oee.round(2))

In [ ]:
# Salva o modelo campeão e as colunas necessárias para inferência na API
caminho_modelo = Path("../models/modelo_preditivo_vulcantech.joblib")
caminho_modelo.parent.mkdir(parents=True, exist_ok=True)

artefato = {
    "model": hgb,
    "feature_names": features,
    "target_classes": PENALIDADES_OEE
}

joblib.dump(artefato, caminho_modelo)
print(f"Modelo e metadados persistidos com sucesso em: {caminho_modelo.resolve()}")